In [36]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression

In [37]:
df = pd.read_csv("../data/taxi_trip_pricing.csv")

df.head()

,Trip_Distance_km,Time_of_Day,Day_of_Week,Passenger_Count,Traffic_Conditions,Weather,Base_Fare,Per_Km_Rate,Per_Minute_Rate,Trip_Duration_Minutes,Trip_Price
0,19.35,Morning,Weekday,3.0,Low,Clear,3.56,0.80,0.32,53.82,36.2624
1,47.59,Afternoon,Weekday,1.0,High,Clear,NaN,0.62,0.43,40.57,NaN
2,36.87,Evening,Weekend,1.0,High,Clear,2.70,1.21,0.15,37.27,52.9032
3,30.33,Evening,Weekday,4.0,Low,NaN,3.48,0.51,0.15,116.81,36.4698
4,NaN,Evening,Weekday,3.0,High,Clear,2.93,0.63,0.32,22.64,15.6180


In [38]:
df["Trip_Price"].isnull().sum()

np.int64(49)

In [39]:
# Dropping Trip Price missing values as if we are not having output itself so how can we train the model on that particular row or for that particular features
df = df.dropna(subset=["Trip_Price"])

In [40]:
df["Trip_Price"].isnull().sum()

np.int64(0)

In [41]:
X = df.drop(columns=["Trip_Price"])
y = df["Trip_Price"]

In [42]:
X.head()

,Trip_Distance_km,Time_of_Day,Day_of_Week,Passenger_Count,Traffic_Conditions,Weather,Base_Fare,Per_Km_Rate,Per_Minute_Rate,Trip_Duration_Minutes
0,19.35,Morning,Weekday,3.0,Low,Clear,3.56,0.80,0.32,53.82
2,36.87,Evening,Weekend,1.0,High,Clear,2.70,1.21,0.15,37.27
3,30.33,Evening,Weekday,4.0,Low,NaN,3.48,0.51,0.15,116.81
4,NaN,Evening,Weekday,3.0,High,Clear,2.93,0.63,0.32,22.64
5,8.64,Afternoon,Weekend,2.0,Medium,Clear,2.55,1.71,0.48,89.33


In [43]:
y.head()

0    36.2624
2    52.9032
3    36.4698
4    15.6180
5    60.2028
Name: Trip_Price, dtype: float64

In [44]:
print("X shape: ", X.shape)
print("y shape: ", y.shape)

X shape:  (951, 10)
y shape:  (951,)


In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [46]:
print("X_train: ", X_train.shape)
print("X_test: ", X_test.shape)

print("y_train: ", y_train.shape)
print("y_test: ", y_test.shape)

X_train:  (760, 10)
X_test:  (191, 10)
y_train:  (760,)
y_test:  (191,)


In [47]:
numerical_features = X_train.select_dtypes(include="number").columns.tolist()

numerical_features

['Trip_Distance_km',
 'Passenger_Count',
 'Base_Fare',
 'Per_Km_Rate',
 'Per_Minute_Rate',
 'Trip_Duration_Minutes']

In [48]:
categorical_features = X_train.select_dtypes(include="str").columns.tolist()

categorical_features

['Time_of_Day', 'Day_of_Week', 'Traffic_Conditions', 'Weather']

In [49]:
# We are using the median strategy for imputer to fill all the nan values because there are some outliers that can make out mean more or less so because of that we are using median
numerical_pipeline = Pipeline([
    ("inputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [50]:
# Here we are using mode/most frequent strategy for imputer as for categorical columns that is the only thing we can do here
# The handle_unknown="ignore" tells us that if in future the column have some another value so it will not throw an error instead it will ignore that row
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [51]:
# Here we are processing the numerical features with the numerical pipeline and the categorical features with the categorical piepeline
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [52]:
# We use fit_transform on the train data because the fit learns the parameters from the data such as mean, std, etc.
X_train_processed = preprocessor.fit_transform(X_train)

In [53]:
# We use transform on the test data because we dont want out test data get the paramteres from the data such as mean, std, etc.
X_test_processed = preprocessor.transform(X_test)

In [54]:
# The number of features of X_train_processed has increased because of the one hot encoding we have used in four categorial features
print("Original X_train shape: ", X_train.shape)
print("Processed X_train shape: ", X_train_processed.shape)

Original X_train shape:  (760, 10)
Processed X_train shape:  (760, 18)


In [55]:
print("Original X_test shape: ", X_test.shape)
print("Processed X_test shape: ", X_test_processed.shape)

Original X_test shape:  (191, 10)
Processed X_test shape:  (191, 18)


In [57]:
model_pipeline = Pipeline([
    ("pre_processor", preprocessor),
    ("model", LinearRegression())
])

In [58]:
model_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('pre_processor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['Trip_Distance_km','Time_of_Day','Day_of_Week',...,'Per_Km_Rate', 'Per_Minute_Rate','Trip_Duration_Minutes']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically

In [59]:
predictions = model_pipeline.predict(X_test)